In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, FloatSlider, Layout


ModuleNotFoundError: No module named 'ipywidgets'

In [13]:
def plot_trident(Lb, Lf, Rs, Re, x_p, y_p, z_p):
    # Setup the figure
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Constants
    P = np.array([x_p, y_p, z_p])
    motor_angles = [0, 120, 240]
    colors = ['#FF4136', '#2ECC40', '#0074D9'] # Red, Green, Blue

    for i, angle in enumerate(motor_angles):
        rad = np.radians(angle)
        # 1. Motor Position (Shoulder)
        M = np.array([Rs * np.cos(rad), Rs * np.sin(rad), 0])

        # 2. Effector Joint Position
        Ej = P + np.array([Re * np.cos(rad), Re * np.sin(rad), 0])

        # 3. Inverse Kinematics for Elbow (E)
        # Vector from Motor to Effector Joint
        V = Ej - M
        d2 = V[0]**2 + V[1]**2 + V[2]**2
        d = np.sqrt(d2)

        # Check if reachable
        if d > (Lb + Lf) or d < abs(Lf - Lb):
            ax.text(0, 0, 0, "OUT OF REACH", color='red', fontsize=20, ha='center')
            continue

        # Solve for elbow in the vertical plane of the motor
        # This simplifies the math for the rotary delta plane
        x_rel = np.sqrt(V[0]**2 + V[1]**2) * np.sign(np.dot(V[:2], M[:2]))
        y_rel = V[2]

        # Circle intersection in 2D
        # (x_rel - Lb*cos(theta))^2 + (y_rel - Lb*sin(theta))^2 = Lf^2
        D = (x_rel**2 + y_rel**2 + Lb**2 - Lf**2) / (2 * Lb * np.sqrt(x_rel**2 + y_rel**2))
        D = np.clip(D, -1, 1)
        theta = np.arctan2(y_rel, x_rel) + np.arccos(D)

        # Elbow in 3D
        E = M + np.array([Lb * np.cos(theta) * np.cos(rad),
                          Lb * np.cos(theta) * np.sin(rad),
                          Lb * np.sin(theta)])

        # Plotting
        ax.plot([M[0], E[0]], [M[1], E[1]], [M[2], E[2]], color=colors[i], linewidth=4, label=f'Motor {i+1}')
        ax.plot([E[0], Ej[0]], [E[1], Ej[1]], [E[2], Ej[2]], color=colors[i], linewidth=2, alpha=0.6)
        ax.scatter(*M, color='black', s=50) # Motor
        ax.scatter(*Ej, color='black', s=20) # Effector Joint

    # Plot the Effector Platform
    effector_pts = []
    for angle in motor_angles + [0]:
        rad = np.radians(angle)
        effector_pts.append(P + np.array([Re * np.cos(rad), Re * np.sin(rad), 0]))
    eff_pts = np.array(effector_pts)
    ax.plot(eff_pts[:,0], eff_pts[:,1], eff_pts[:,2], color='purple', linewidth=2)
    ax.scatter(*P, color='purple', s=100) # Nozzle

    # Aesthetics
    ax.set_xlim(-400, 400); ax.set_ylim(-400, 400); ax.set_zlim(-600, 100)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=20, azim=45)
    plt.title(f"R-Trident Live Sim: Lb={Lb}, Lf={Lf}")
    plt.show()

# --- INTERACTIVE WIDGETS ---
interact(plot_trident,
    Lb=FloatSlider(min=50, max=300, step=5, value=150, description='Bicep (Lb)'),
    Lf=FloatSlider(min=100, max=600, step=5, value=300, description='Forearm (Lf)'),
    Rs=FloatSlider(min=50, max=400, step=5, value=150, description='Shoulder (Rs)'),
    Re=FloatSlider(min=10, max=100, step=5, value=50, description='Effector (Re)'),
    x_p=FloatSlider(min=-200, max=200, step=1, value=0, description='X Nozzle'),
    y_p=FloatSlider(min=-200, max=200, step=1, value=0, description='Y Nozzle'),
    z_p=FloatSlider(min=-600, max=-100, step=5, value=-300, description='Z Nozzle')
)

interactive(children=(FloatSlider(value=150.0, description='Bicep (Lb)', max=300.0, min=50.0, step=5.0), Float…

<function __main__.plot_trident(Lb, Lf, Rs, Re, x_p, y_p, z_p)>